<span style="font-family:monospace;letter-spacing:.14em;color:#857259">BANA · MODULE 2 · EXPLORING AND VISUALISING DATA</span>

# Session 1: Meet the data

**Dataset:** EV purchase intention (Kaggle Playground Series S6E9, a *synthetic* dataset). You work with a 10,000-row seeded sample, `ev_m2_sample.csv`.
**Time:** 60 minutes. **Course outcomes:** CO2 (and CO3 for Session 5).

| Block | Min | What you do |
|---|---|---|
| Hook | 5 | Guess the share of people who say Yes |
| Concept | 10 | Rows, columns, target, base rate (instructor) |
| Guided lab | 30 | Inspect, document, derive four columns |
| Team task | 10 | Three questions a manager would ask |
| Exit ticket | 5 | Three quick checks |

**By the end of this session you can:**
- say what one row is and what we are trying to predict
- name the type of every column (number, category, ordered category)
- build four derived columns that every later module reuses
- state the base rate and why it matters

**New commands this session (max 5):** `head()`, `shape`, `info()`, `isna().sum()`, `value_counts()`

**AI-use (AMBER protocol).** You may use an AI tool to look up *syntax*. The reasoning and interpretation must be your own: you will defend them in the viva.

## Facilitator notes: Session 1
**Question for the room:** *Who is one row, and what are we trying to predict?*

**Run of show**
1. **Hook (5):** Cold-call five students for their guess of "out of 100 people, how many say they will buy an EV?" Write guesses on the board. Reveal after Part B. Truth in this sample: **18.2%** Yes (full 668,665-row file: 17.46%).
2. **Concept (10):** unit of analysis, target vs features, variable types, base rate, CRISP-DM Data Understanding (Chapman et al., 2000). Say plainly that this dataset is **synthetic**: produced for a Kaggle Playground competition from a smaller original dataset, so patterns are cleaner than real survey data. This is a lineage lesson, not a flaw.
3. **Lab (30):** Parts B to E. Pair a strong and a weak coder.
4. **Team task (10):** three manager questions, each must name a column.
5. **Exit ticket (5):** answers at the end of this notebook.

**Traps to expect**
- The sample has **blanks that are not in the Kaggle data**. The instructor injected them for Session 3 (488 in income, 165 in commute, 194 in environmental concern). Students find them with `isna().sum()`; do not explain why yet.
- Students read `Range_Anxiety_Level` as a number. It is an *ordered category* (Low, Medium, High).
- Accuracy of the lazy rule "everyone says No" is **81.8%**. Plant this; it returns in Module 5.

In [1]:
# -- Setup: run once ------------------------------------------------------
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
warnings.filterwarnings("ignore")

# House style: Nikhil D Jonathan design system (sand ground, terracotta / lagoon / ochre data colours)
C = {"lagoon": "#25707B", "terracotta": "#B5562F", "ochre": "#E0A231", "moss": "#3F6F4B",
     "earth": "#665643", "ink": "#221B15", "sand": "#F8F3EA", "card": "#FFFDF9", "line": "#E3D6C1"}
sns.set_theme(style="whitegrid", rc={
    "figure.facecolor": C["sand"], "axes.facecolor": C["card"], "axes.edgecolor": "#CDBCA3",
    "grid.color": C["line"], "text.color": C["ink"], "axes.labelcolor": C["ink"],
    "xtick.color": "#4A3E31", "ytick.color": "#4A3E31", "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 100, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.titlelocation": "left"})
PCT = PercentFormatter(1, decimals=0)                              # axis labels as percentages
SEQ = sns.light_palette(C["terracotta"], as_cmap=True)   # one-hue sequential map for heatmaps

# -- Load the data (Colab: upload ev_m2_sample.csv to the Files pane, or run the upload below)
FILE = "ev_m2_sample.csv"
if not os.path.exists(FILE):
    try:
        from google.colab import files
        print("Choose ev_m2_sample.csv"); files.upload()
    except ImportError:
        raise FileNotFoundError("Put ev_m2_sample.csv in the same folder as this notebook.")
raw = pd.read_csv(FILE)
df = raw.copy()
print("Loaded", df.shape[0], "rows x", df.shape[1], "columns")

Loaded 10000 rows x 15 columns


## Part A. Hook: guess first (5 min)
Out of **100** people in this dataset, how many do you think say they *will buy an EV*? Write one number. Do not look at the data yet.

In [2]:
my_guess = 18   # example answer

## Part B. Look before you touch (10 min)
**Predict, then run.** Before each cell, write your prediction in a comment, then run it.

In [3]:
df.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,61,34,95772.0,22.0,1,1,1,3.0,Female,Rural,Sedan,No,No,Medium,No
1,197,53,106533.0,24.2,1,3,6,2.0,Male,Suburban,Sedan,Yes,Yes,Low,No
2,246,63,73220.0,17.6,2,1,0,4.0,Female,Rural,Sedan,No,Yes,Medium,No
3,333,69,30000.0,5.0,2,0,0,3.0,Female,Rural,Sedan,Yes,No,Low,No
4,518,52,76983.0,50.8,1,2,2,4.0,Female,Rural,Sedan,Yes,No,Low,No


In [4]:
df.shape      # (rows, columns)

(10000, 15)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id                           10000 non-null  int64  
 1   Age                          10000 non-null  int64  
 2   Annual_Income_USD            9512 non-null   float64
 3   Daily_Commute_km             9835 non-null   float64
 4   Number_of_Cars_Owned         10000 non-null  int64  
 5   Charging_Stations_Near_Home  10000 non-null  int64  
 6   Charging_Stations_Near_Work  10000 non-null  int64  
 7   Environmental_Concern_Level  9806 non-null   float64
 8   Gender                       10000 non-null  str    
 9   City_Type                    10000 non-null  str    
 10  Current_Car_Type             10000 non-null  str    
 11  Home_Charging_Possible       10000 non-null  str    
 12  Subsidy_Available            10000 non-null  str    
 13  Range_Anxiety_Level         

In [6]:
df.isna().sum()      # blanks per column

id                               0
Age                              0
Annual_Income_USD              488
Daily_Commute_km               165
Number_of_Cars_Owned             0
Charging_Stations_Near_Home      0
Charging_Stations_Near_Work      0
Environmental_Concern_Level    194
Gender                           0
City_Type                        0
Current_Car_Type                 0
Home_Charging_Possible           0
Subsidy_Available                0
Range_Anxiety_Level              0
Will_Buy_EV                      0
dtype: int64

**YOUR ANSWER:** Which columns have blanks, and roughly what share of rows?

> _write here_

**Expected:** Annual_Income_USD (4.9%), Daily_Commute_km (1.7%), Environmental_Concern_Level (1.9%). All others are complete. *These blanks were injected by the instructor for teaching.*

In [7]:
df['Will_Buy_EV'].value_counts()

Will_Buy_EV
No     8181
Yes    1819
Name: count, dtype: int64

In [8]:
df['Will_Buy_EV'].value_counts(normalize=True).round(3)      # shares instead of counts

Will_Buy_EV
No     0.818
Yes    0.182
Name: proportion, dtype: float64

**Reveal.** Compare with your guess in Part A. How far off were you? That Yes share is called the **base rate**.

## Part C. Build the data dictionary (8 min)
For each column, fill the `kind` cell with one of: `number`, `category`, `ordered category`, `yes/no`, `id`.

In [9]:
dd = pd.DataFrame({
    "column": df.columns,
    "kind": ["id", "number", "number", "number", "number", "number", "number", "ordered category",
             "category", "category", "category", "yes/no", "yes/no", "ordered category", "yes/no (target)"],
    "plain_meaning": [
        "Row identifier", "Age in years", "Annual household income in US dollars",
        "One-way daily commute in km", "Cars owned by the household",
        "Public charging stations near home", "Public charging stations near work",
        "Self-rated environmental concern, 1 (low) to 5 (high)", "Gender",
        "Urban, Suburban or Rural", "Type of current car", "Can the person charge at home?",
        "Is a purchase subsidy available?", "Worry about running out of range",
        "Does the person say they WILL buy an EV?"],
})
dd

,column,kind,plain_meaning
0,id,id,Row identifier
1,Age,number,Age in years
2,Annual_Income_USD,number,Annual household income in US dollars
3,Daily_Commute_km,number,One-way daily commute in km
4,Number_of_Cars_Owned,number,Cars owned by the household
5,Charging_Stations_Near_Home,number,Public charging stations near home
6,Charging_Stations_Near_Work,number,Public charging stations near work
7,Environmental_Concern_Level,ordered category,"Self-rated environmental concern, 1 (low) to 5..."
8,Gender,category,Gender
9,City_Type,category,"Urban, Suburban or Rural"


> Note: the dataset does not say whether *Annual_Income_USD* is per person or per household, or which country. **A good analyst writes down what is unknown.** Add one line under the table listing two things you cannot tell from the data.

## Part D. Build four derived columns (7 min)
These four columns are reused in **every** later module. Fill the one `TODO`; read the rest.

In [10]:
def add_derived(d):
    """The four derived columns built in Session 1 (pre-written so every notebook stands alone)."""
    d = d.copy()
    d["buy"] = (d["Will_Buy_EV"] == "Yes").astype(int)
    d["age_band"] = pd.cut(d["Age"], bins=[24, 34, 44, 54, 69], labels=["25-34", "35-44", "45-54", "55-69"])
    d["income_band"] = pd.qcut(d["Annual_Income_USD"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
    d["charger_total"] = d["Charging_Stations_Near_Home"] + d["Charging_Stations_Near_Work"]
    return d

In [11]:
df = add_derived(df)
df[['Will_Buy_EV', 'buy', 'Age', 'age_band', 'Annual_Income_USD', 'income_band', 'charger_total']].head()

,Will_Buy_EV,buy,Age,age_band,Annual_Income_USD,income_band,charger_total
0,No,0,34,25-34,95772.0,Q3,2
1,No,0,53,45-54,106533.0,Q4 highest,9
2,No,0,63,55-69,73220.0,Q2,1
3,No,0,69,55-69,30000.0,Q1 lowest,0
4,No,0,52,45-54,76983.0,Q2,4


## Part E. The base rate and the lazy rule (5 min)

In [12]:
base_rate = df["buy"].mean()
lazy_accuracy = 1 - base_rate
print(f"Base rate of Yes: {base_rate:.1%}")
print(f"If we predict 'No' for everyone we are right {lazy_accuracy:.1%} of the time")

Base rate of Yes: 18.2%
If we predict 'No' for everyone we are right 81.8% of the time


**YOUR ANSWER:** A model that is 'right' about four times in five sounds good. Why is this rule useless to an EV marketing team?

> _write here_

**Expected answer:** it never finds a single buyer, so it cannot target anyone. High accuracy can hide zero usefulness when one class is rare. (Returns in Module 5: confusion matrix, recall.)

## Part F. Where does this data come from? (data lineage)
Write one line for each.

- **Who or what does a row represent?** 
- **How was it collected?** 
- **What is missing from the story?** 

> _write here_

**Expected:** a row is a *synthetic* potential car buyer, generated by a model for a Kaggle Playground competition (Series 6, Episode 9) from a smaller original dataset of 10,000 synthetic buyers. No real person is in the file. Consequences: (1) patterns are cleaner and stronger than real surveys, so never quote these percentages as market facts; (2) privacy risk is low, but the *method* is the same one used on real customer data; (3) income currency/country is not documented, so any India framing is an exercise, not a finding.
**Optional stretch:** if you have the full Kaggle `train.csv` (668,665 rows) and `test.csv` (286,571 rows, no target column), compare Age and Income means across the two files: they match closely (47.04 vs 47.07; 84,769 vs 84,800), which tells you the test file was drawn from the same source.

## Team task (10 min)
Write **three questions** an EV marketing manager would ask of this data. Each must name at least one column and say what you would compare.

| # | Question | Column(s) | What I would compare |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

**Strong examples to praise:** "Does a subsidy change who says Yes?" (Subsidy_Available vs buy: 28.5% vs 0.6%). "Do rural buyers differ from urban?" (City_Type). "Is concern for the environment enough on its own?" (Environmental_Concern_Level). **Weak:** "Who will buy?" (no column, no comparison).

## Exit ticket (5 min)
1. In one sentence: what is one row in this dataset?
2. Name the target column and the base rate.
3. Name one column that is an *ordered category* and say why it is not a plain number.

**Answers:** (1) one synthetic potential buyer; (2) `Will_Buy_EV` / `buy`, 18.2% Yes in this sample; (3) `Range_Anxiety_Level` (Low < Medium < High) or `Environmental_Concern_Level`: the gaps between levels are not equal numbers.

**Viva prompt (AMBER):** "Show me your `buy` column and tell me how you know it is coded the right way round."

### References (facilitator only)
Chapman, P., et al. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS/CRISP-DM Consortium.  
GAISE College Report ASA Revision Committee (2016). *Guidelines for Assessment and Instruction in Statistics Education: College Report 2016*. American Statistical Association.  
Kirschner, P. A., Sweller, J., & Clark, R. E. (2006). Why minimal guidance during instruction does not work. *Educational Psychologist*, 41(2), 75-86.
Wickham, H. (2014). Tidy data. *Journal of Statistical Software*, 59(10), 1-23.
Provost, F., & Fawcett, T. (2013). *Data Science for Business*. O'Reilly.
Kaggle (2026). *Playground Series, Season 6, Episode 9: Predicting Electric Vehicle Purchases*. kaggle.com/competitions/playground-series-s6e9